# 🧱 Baseline Models

**✍️ Author:** Hayriye Anıl  
**📘 Blog Series:** Time Series Analysis & Forecasting  

## 🎯 Purpose

This notebook implements and evaluates baseline forecasting models for temperature prediction. Baseline models serve as performance benchmarks that more sophisticated models should outperform. We implement four simple yet effective baseline approaches and compare their forecasting accuracy.

## 📂 Contents

1. **Data Preparation**
   - Load processed weather dataset
   - Split data into training and test sets (2025-2026 as test period)

2. **Baseline Model Implementation**
   - **Arithmetic Mean**: Uses historical average as constant prediction
   - **Rolling Mean**: Adapts predictions using moving average window
   - **Naive 1-Hour**: Simple persistence model (previous hour's value)
   - **Naive 24-Hour**: Seasonal naive model (same time yesterday)

3. **Model Evaluation**
   - Calculate performance metrics (MAE, RMSE, Bias, NMAE, NRMSE)
   - Compare baseline models using visualization
   - Establish performance benchmarks for future model comparison

4. **Multi-Horizon Forecasting**
   - Extend baseline models for multiple forecast horizons (H+24, H+48, H+72, H+96, H+120)
   - Evaluate performance degradation across different forecast distances


In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

import pandas as pd

from data.paths import PROCESSED_DATA_DIR, BASELINE_MODEL_DIR, PERFORMANCE_DIR
from src import baseline_models as bm
from src import kpis as kpi
from src import plots as plot

### Custom Functions

In [ ]:
def reshape_metrics_tables(multihorizon_metrics_frame: pd.DataFrame):
    """Reshape multihorizon metrics dataframe into a melted format.
    
    Transforms a dataframe with index format 'H+XX_model_name' into a long format
    with separate columns for model, horizon, and all metrics.
    
    Args:
        multihorizon_metrics_frame : DataFrame with index containing horizon and model info 
        (e.g., 'H+24_arithmetic_mean') and columns containing metric values (MAE (°C), RMSE (°C), etc.)
    
    Returns
    -------
    pd.DataFrame
        Melted dataframe with columns: model, horizon, MAE (°C), RMSE (°C), 
        Bias (°C), NMAE, NRMSE.
    """
    df = multihorizon_metrics_frame.copy()
    split_index = df.index.str.split('_')
    df['horizon'] = split_index.str[0]
    df['model'] = split_index.str[1:].str.join('_')  
    
    # Melt the dataframe to get Model | Horizon | Metric columns
    melted = df.melt(
        id_vars=['model', 'horizon'], 
        var_name='metric', 
        value_name='value'
    )
    
    result = melted.pivot_table(
        index=['model', 'horizon'], 
        columns='metric', 
        values='value'
    ).reset_index()
    result.columns.name = None
    column_order = ['model', 'horizon', 'MAE (°C)', 'RMSE (°C)', 'Bias (°C)', 'NMAE', 'NRMSE']
    result = result[column_order]
    model_order = ['arithmetic_mean', 'rolling_mean', 'naive_1h', 'naive_24h']
    horizon_order = ['H+24', 'H+48', 'H+72', 'H+96', 'H+120']
    
    result['model'] = pd.Categorical(result['model'], categories=model_order, ordered=True)
    result['horizon'] = pd.Categorical(result['horizon'], categories=horizon_order, ordered=True)
    
    return result.sort_values(['model', 'horizon']).reset_index(drop=True)

In [ ]:
dataset = pd.read_csv(f"{PROCESSED_DATA_DIR}/processed_dataset.csv", 
                      index_col=0, 
                      parse_dates=True)
dataset

In [ ]:
data = dataset[["temperature_2m (°C)"]]
data

In [ ]:
train_set = data[data.index < '2025-01-01']
test_set = data[(data.index >= '2025-01-01') & (data.index <= '2026-01-01')] # 1 year test set

In [ ]:
train_set

In [ ]:
test_set

### Baseline Models

In [ ]:
arithmetic_mean_baseline_df = bm.baseline_arithmetic_mean_model(train_set,
                                                                test_set,
                                                                target="temperature_2m (°C)",
                                                                window_size=30 * 24)
arithmetic_mean_baseline_df

In [ ]:
rolling_mean_baseline_df = bm.baseline_rolling_mean_model(data,
                                                          test_set,
                                                          target="temperature_2m (°C)",
                                                          window_size = 30 * 24)
rolling_mean_baseline_df

In [ ]:
naive_1h_baseline_df = bm.baseline_model_seasonal_naive(test_set,
                                                        target="temperature_2m (°C)",
                                                        period=1)
naive_1h_baseline_df

In [ ]:
naive_24h_baseline_df = bm.baseline_model_seasonal_naive(test_set, 
                                                         target="temperature_2m (°C)", 
                                                         period=24)
naive_24h_baseline_df

In [ ]:
# Concatenate predictions from all baseline models with actual temperature values
# to enable direct comparison of forecast accuracy across models for the same time periods
baseline_models_forecasts = pd.concat([
    arithmetic_mean_baseline_df, 
    rolling_mean_baseline_df["rolling_mean_pred"], 
    naive_1h_baseline_df["H+1"],
    naive_24h_baseline_df["H+24"]
], axis=1)
baseline_models_forecasts.dropna(inplace=True)
baseline_models_forecasts

In [ ]:
plot.plot_line(baseline_models_forecasts,
               baseline_models_forecasts.index,
               baseline_models_forecasts.columns,
               "Time",
               "°C",
               "All Baseline Models vs Actuals")

In [ ]:
metrics = []
for column in baseline_models_forecasts.columns[1:]:
    metric = kpi.metrics_summary(baseline_models_forecasts,
                                  actual="temperature_2m (°C)",
                                  pred=column)
    metrics.append(metric)
    

all_baseline_metrics = pd.concat(metrics)
all_baseline_metrics.index = ["Arithmetic Mean", "Rolling Mean", "Naive 1-Hour", "Naive 24-Hour"]
all_baseline_metrics

In [ ]:
plot.plot_bar(all_baseline_metrics, all_baseline_metrics.index, 
             all_baseline_metrics["NMAE"],
             "Baseline Models",
             "NMAE",
             "NMAE of Baseline Models")

In [ ]:
plot.plot_bar(all_baseline_metrics, all_baseline_metrics.index, 
             all_baseline_metrics["MAE (°C)"],
             "Baseline Models",
             "MAE (°C)",
             "MAE of Baseline Models")

#### Visuals for Baseline Models Comparison Between Forecast and Actual

In [ ]:
plot.plot_line(arithmetic_mean_baseline_df,
               arithmetic_mean_baseline_df.index,
               arithmetic_mean_baseline_df.columns,
               "Time",
               "°C",
               "Arithmetic Mean Baseline Model: Forecast vs Actual")

In [ ]:
plot.plot_line(rolling_mean_baseline_df,
               rolling_mean_baseline_df.index,
               rolling_mean_baseline_df.columns,
               "Time",
               "°C",
               "Rolling Mean Baseline Model: Forecast vs Actual")

In [ ]:
plot.plot_line(naive_1h_baseline_df,
               naive_1h_baseline_df.index,
               naive_1h_baseline_df.columns,
               "Time",
               "°C",
               "Naive 1-Hour Baseline Model: Forecast vs Actual")

In [ ]:
plot.plot_line(naive_24h_baseline_df,
               naive_24h_baseline_df.index,
               naive_24h_baseline_df.columns,
               "Time",
               "°C",
               "Seasonal Naive 24-Hour Baseline Model: Forecast vs Actual")

### Multi-Horizon Baseline Models

In [ ]:
forecast_days = 5
timestamps_per_day = 24
forecast_horizon = forecast_days * timestamps_per_day  # 120

In [ ]:
arithmetic_mean_multihorizon = bm.baseline_arithmetic_mean_for_multihorizon(train_set,
                                                                            test_set,
                                                                            target="temperature_2m (°C)",
                                                                            window_size=30 * 24,
                                                                            forecast_days=forecast_days,
                                                                            timestamps_per_day=timestamps_per_day)
arithmetic_mean_multihorizon.to_csv(f"{BASELINE_MODEL_DIR}/arithmetic_mean_multihorizon_forecasts.csv")
arithmetic_mean_multihorizon

In [ ]:
rolling_mean_multihorizon = bm.baseline_rolling_mean_for_multihorizon(data,
                                                                      test_set,
                                                                      target="temperature_2m (°C)",
                                                                      window_size = 30 * 24,
                                                                      forecast_days=forecast_days,
                                                                      timestamps_per_day=timestamps_per_day)
rolling_mean_multihorizon.to_csv(f"{BASELINE_MODEL_DIR}/rolling_mean_multihorizon_forecasts.csv")
rolling_mean_multihorizon

In [ ]:
naive_1h_multihorizon = bm.baseline_model_seasonal_naive_1h_for_multihorizon(test_set,
                                                                             target="temperature_2m (°C)",
                                                                             forecast_days=forecast_days,
                                                                             timestamps_per_day=timestamps_per_day)
naive_1h_multihorizon.to_csv(f"{BASELINE_MODEL_DIR}/naive_1h_multihorizon_forecasts.csv")
naive_1h_multihorizon

In [ ]:
naive_24h_multihorizon = bm.baseline_model_seasonal_naive_24h_for_multihorizon(test_set, 
                                                                               target="temperature_2m (°C)", 
                                                                               forecast_days=forecast_days,
                                                                               timestamps_per_day=timestamps_per_day)
naive_24h_multihorizon.to_csv(f"{BASELINE_MODEL_DIR}/naive_24h_multihorizon_forecasts.csv")
naive_24h_multihorizon

In [ ]:
columns = ['H+24', 'H+48', 'H+72', 'H+96', 'H+120']
target = "temperature_2m (°C)"

models = {
    "arithmetic_mean": arithmetic_mean_multihorizon,
    "rolling_mean":    rolling_mean_multihorizon,
    "naive_1h":        naive_1h_multihorizon,
    "naive_24h":       naive_24h_multihorizon,
}

multihorizon_metrics = []

for column in columns:
    data_frame = pd.DataFrame({target: models["arithmetic_mean"][target]}) 
    for name, mdf in models.items():
        data_frame[f"{column}_{name}"] = mdf[column]

    data_frame = data_frame.dropna()
    
    # Compute metrics for each model column
    rows = []
    for name in models.keys():
        pred_col = f"{column}_{name}"
        metric = kpi.metrics_summary(data_frame, actual=target, pred=pred_col)
        metric.index = [f"{column}_{name}"]  # nice row label
        rows.append(metric)

    multihorizon_metrics.append(pd.concat(rows))

multihorizon_metrics_frame = pd.concat(multihorizon_metrics)
multihorizon_metrics_frame

In [ ]:
metric_table = reshape_metrics_tables(multihorizon_metrics_frame)
metric_table

In [ ]:
metric_table.to_csv(f"{PERFORMANCE_DIR}/baseline_models_multihorizon_metrics.csv", index=False)

In [ ]:
for column in metric_table.columns[2:]:
    plot.plot_bar_group(metric_table,
                        "model",
                        column,
                        "horizon",
                        column,
                        f"{column} Scores by Model and Horizon")

#### Visuals for Multi-Horizon Baseline Models Comparison Between Forecast and Actual

In [ ]:
plot.plot_line(arithmetic_mean_multihorizon,
               arithmetic_mean_multihorizon.index,
               arithmetic_mean_multihorizon.columns,
               "Time",
               "°C",
               "Multi-Horizon Arithmetic Mean Baseline Model vs Actuals")

In [ ]:
plot.plot_line(rolling_mean_multihorizon,
               rolling_mean_multihorizon.index,
               rolling_mean_multihorizon.columns,
               "Time",
               "°C",
               "Multi-Horizon Rolling Mean Baseline Model vs Actual")

In [ ]:
plot.plot_line(naive_1h_multihorizon,
               naive_1h_multihorizon.index,
               naive_1h_multihorizon.columns,
               "Time",
               "°C",
               "Multi-Horizon Naive 1-Hour Baseline Model vs Actual")

In [ ]:
plot.plot_line(naive_24h_multihorizon,
               naive_24h_multihorizon.index,
               naive_24h_multihorizon.columns,
               "Time",
               "°C",
               "Multi-Horizon Seasonal Naive 24-Hour Baseline Model vs Actual")

### Validation Check: Metric Calculation Consistency

Issue: The baseline model metrics were calculated using the entire test period (2025-01-02 to 2025-12-31), but this includes different amounts of data for each forecast horizon due to shifting.

**Problem:**
- H+24 forecasts start 24 hours later than H+120 forecasts
- Different horizons have different valid date ranges after dropping NaN values
- Comparing metrics across horizons may be misleading due to different evaluation periods

**Solution:** Recalculate H+24 metrics using only the date range where H+24 predictions are actually available, then compare with the original H+24 metrics to verify consistency.

**Expected Result:** If calculations are correct, the recalculated H+24 metrics should match the original H+24 metrics exactly, confirming our evaluation methodology is sound.

This validation ensures we're making fair comparisons between different forecast horizons.

In [ ]:
horizon_first_day_forecasts = pd.concat([
    arithmetic_mean_multihorizon[["temperature_2m (°C)", "H+24"]].rename(columns={"H+24":"H+24 Arithmetic Mean"}),
    rolling_mean_multihorizon["H+24"].rename("H+24 Rolling Mean", inplace=True),
    naive_1h_multihorizon["H+24"].rename("H+24 Naive 1-Hour", inplace=True),
    naive_24h_multihorizon["H+24"].rename("H+24 Naive 24-Hour", inplace=True)
    
], axis=1)
horizon_first_day_forecasts.dropna(inplace=True)
horizon_first_day_forecasts

In [ ]:
horizon_first_day_forecasts.index[0], horizon_first_day_forecasts.index[-1]

In [ ]:
testing_first_horizon_forecast = baseline_models_forecasts[(baseline_models_forecasts.index >= horizon_first_day_forecasts.index[0]) & (baseline_models_forecasts.index <= horizon_first_day_forecasts.index[-1]) ]
testing_first_horizon_forecast

In [ ]:
metrics = []
for column in testing_first_horizon_forecast.columns[1:]:
    metric = kpi.metrics_summary(testing_first_horizon_forecast,
                                  actual="temperature_2m (°C)",
                                  pred=column)
    metrics.append(metric)
    

testing_first_horizon_metrics = pd.concat(metrics)
testing_first_horizon_metrics.index = ["H+24 Arithmetic Mean", "H+24 Rolling Mean", "H+24 Naive 1-Hour", "H+24 Naive 24-Hour"]
testing_first_horizon_metrics

In [ ]:
first_horizon_metric = multihorizon_metrics_frame.loc[
        multihorizon_metrics_frame.index.str.startswith("H+24")]
first_horizon_metric